## 추천 시스템

In [2]:
import pandas as pd

etf_data = [
    {
        "name": "KODEX 200",
        "category": "국내주식",
        "expense_ratio": 0.15,
        "return_1y": 8.5,
        "risk_level": "중간",
        "dividend_yield": 1.8,
        "volatility": 15.2,
    },
    {
        "name": "KODEX 미국S&P500TR",
        "category": "해외주식",
        "expense_ratio": 0.05,
        "return_1y": 25.3,
        "risk_level": "중간",
        "dividend_yield": 0.0,
        "volatility": 18.5,
    },
    {
        "name": "ACE 미국배당다우존스",
        "category": "배당",
        "expense_ratio": 0.01,
        "return_1y": 12.1,
        "risk_level": "낮음",
        "dividend_yield": 3.5,
        "volatility": 10.3,
    },
    {
        "name": "TIGER 2차전지테마",
        "category": "테마",
        "expense_ratio": 0.45,
        "return_1y": -15.2,
        "risk_level": "높음",
        "dividend_yield": 0.0,
        "volatility": 35.7,
    },
    {
        "name": "TIGER 국고채10년",
        "category": "채권",
        "expense_ratio": 0.07,
        "return_1y": 4.2,
        "risk_level": "낮음",
        "dividend_yield": 2.8,
        "volatility": 5.1,
    },
    {
        "name": "KODEX 골드선물(H)",
        "category": "원자재",
        "expense_ratio": 0.68,
        "return_1y": 18.7,
        "risk_level": "중간",
        "dividend_yield": 0.0,
        "volatility": 20.4,
    },
    {
        "name": "KODEX 미국나스닥100TR",
        "category": "해외주식",
        "expense_ratio": 0.05,
        "return_1y": 32.1,
        "risk_level": "높음",
        "dividend_yield": 0.0,
        "volatility": 25.3,
    },
    {
        "name": "TIGER 미국반도체",
        "category": "섹터",
        "expense_ratio": 0.49,
        "return_1y": 45.0,
        "risk_level": "높음",
        "dividend_yield": 0.0,
        "volatility": 38.2,
    },
    {
        "name": "KODEX 단기채권PLUS",
        "category": "채권",
        "expense_ratio": 0.03,
        "return_1y": 3.8,
        "risk_level": "낮음",
        "dividend_yield": 3.2,
        "volatility": 1.5,
    },
    {
        "name": "TIGER 리츠부동산",
        "category": "리츠",
        "expense_ratio": 0.29,
        "return_1y": 6.5,
        "risk_level": "중간",
        "dividend_yield": 4.8,
        "volatility": 12.8,
    },
    {
        "name": "KODEX 200TR",
        "category": "국내주식",
        "expense_ratio": 0.12,
        "return_1y": 9.1,
        "risk_level": "중간",
        "dividend_yield": 0.0,
        "volatility": 14.9,
    },
    {
        "name": "TIGER 고배당저변동",
        "category": "배당",
        "expense_ratio": 0.30,
        "return_1y": 7.2,
        "risk_level": "낮음",
        "dividend_yield": 5.2,
        "volatility": 8.7,
    },
]

etf_df = pd.DataFrame(etf_data)
etf_df.head()


,name,category,expense_ratio,return_1y,risk_level,dividend_yield,volatility
0,KODEX 200,국내주식,0.15,8.5,중간,1.8,15.2
1,KODEX 미국S&P500TR,해외주식,0.05,25.3,중간,0.0,18.5
2,ACE 미국배당다우존스,배당,0.01,12.1,낮음,3.5,10.3
3,TIGER 2차전지테마,테마,0.45,-15.2,높음,0.0,35.7
4,TIGER 국고채10년,채권,0.07,4.2,낮음,2.8,5.1


In [ ]:
import numpy as np

risk_map = {"낮음": 1, "중간": 2, "높음": 3}


def etf_to_vector(etf):
    return np.array(
        [
            risk_map[etf["risk_level"]] / 3,
            (etf["return_1y"] + 30) / 80,
            etf["dividend_yield"] / 6,
            etf["expense_ratio"],
            etf["volatility"] / 40,
        ]
    )


item_vectors = np.array([etf_to_vector(etf) for etf in etf_data])
item_names = [etf["name"] for etf in etf_data]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(item_vectors)


def cbf_similar_items(target_name, top_k=5):
    target_idx = item_names.index(target_name)
    sim_scores = list(enumerate(item_sim[target_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i[0] for i in sim_scores[1 : top_k + 1]]
    return [
        (item_names[i], round(float(item_sim[target_idx][i]), 2)) for i in top_indices
    ]


cbf_similar_items("KODEX 200")

[('KODEX 200TR', 0.95),
 ('KODEX 미국나스닥100TR', 0.94),
 ('KODEX 미국S&P500TR', 0.93),
 ('TIGER 미국반도체', 0.92),
 ('TIGER 리츠부동산', 0.91)]

In [ ]:
def cbf_diverse(target_name, top_k=5):
    target_idx = item_names.index(target_name)
    sim_scores = list(enumerate(item_sim[target_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    categories = set()
    top_indices = []
    for [idx, score] in sim_scores:
        if len(top_indices) >= top_k:
            break
        category = etf_data[idx]["category"]
        if category in categories:
            continue
        categories.add(category)
        score = round(float(item_sim[target_idx][idx]), 2)
        name = item_names[idx]
        top_indices.append((name, score, category))
    return top_indices


cbf_diverse("KODEX 200")

[('KODEX 200', 1.0, '국내주식'),
 ('KODEX 미국나스닥100TR', 0.94, '해외주식'),
 ('TIGER 미국반도체', 0.92, '섹터'),
 ('TIGER 리츠부동산', 0.91, '리츠'),
 ('TIGER 국고채10년', 0.89, '채권')]